# 05. Build the Power BI feed

**Purpose:** Create the Power BI-ready tables and views from the SQL file `sql/16_powerbi_views.sql`, check them against reference numbers, and export them as CSV files for the dashboard.

**Inputs:** `vw_order_base`, `vw_seller_scorecard`, and the `seller_scores` table created by notebook 04.

**Outputs:** `dim_state`, `vw_pbi_orders`, `vw_pbi_order_category`, `vw_pbi_seller` in MySQL; CSV exports in `powerbi/data/` (not committed).

**Run order:** after notebook 04.

## 1. Setup

In [1]:
import os
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent
load_dotenv(ROOT / ".env")
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{quote_plus(os.getenv('DB_PASSWORD'))}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}?charset=utf8mb4"
)

def run(sql):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

n_orders = run("SELECT COUNT(*) FROM vw_order_base").iloc[0, 0]
assert n_orders == 99441, "vw_order_base should have one row per order"
print(f"vw_order_base rows: {n_orders:,}")

vw_order_base rows: 99,441


## 2. Check the dependencies

The seller view reads the `seller_scores` table written by notebook 04.

In [2]:
seller_scores_exists = run("SHOW TABLES LIKE 'seller_scores'").shape[0] == 1
assert seller_scores_exists, "Run notebook 04 first: it creates the seller_scores table"
print("seller_scores table exists:", seller_scores_exists)
print("seller scorecard view rows:", run("SELECT COUNT(*) FROM vw_seller_scorecard").iloc[0, 0])
print(run("SELECT tier, COUNT(*) AS sellers FROM seller_scores GROUP BY tier"))

seller_scores table exists: True
seller scorecard view rows: 2965
       tier  sellers
0  Reliable      376
1   At risk       93
2     Watch      157


## 3. Run the SQL file one statement at a time

In [3]:
sql_text = (ROOT / "sql" / "16_powerbi_views.sql").read_text(encoding="utf-8")
no_comments = "\n".join(l for l in sql_text.splitlines() if not l.strip().startswith("--"))
statements = [s.strip() for s in no_comments.split(";") if s.strip()]
print(len(statements), "statements found")

for i, stmt in enumerate(statements, 1):
    with engine.begin() as conn:
        conn.exec_driver_sql(stmt)
    print(f"{i}/{len(statements)} OK:", stmt.split("\n")[0][:80])

9 statements found
1/9 OK: USE olist
2/9 OK: DROP TABLE IF EXISTS dim_state
3/9 OK: CREATE TABLE dim_state (
4/9 OK: INSERT INTO dim_state (state_code, state_name, region) VALUES
5/9 OK: UPDATE dim_state SET map_location = CONCAT(state_name, ', Brazil')
6/9 OK: CREATE OR REPLACE VIEW vw_pbi_orders AS
7/9 OK: CREATE OR REPLACE VIEW vw_pbi_order_category AS
8/9 OK: CREATE OR REPLACE VIEW vw_pbi_seller AS
9/9 OK: SELECT COUNT(*)                                   AS orders,


## 4. Reference check

The dashboard cards must match these numbers.

In [4]:
assert run("SELECT COUNT(*) FROM vw_pbi_orders").iloc[0, 0] == 99441
for name in ["vw_pbi_orders", "vw_pbi_order_category", "vw_pbi_seller", "dim_state"]:
    print(f"{name:24s}", run(f"SELECT COUNT(*) FROM {name}").iloc[0, 0])

display(run("""
    SELECT COUNT(*)                                   AS orders,
           SUM(is_active_order)                       AS active_orders,
           ROUND(SUM(CASE WHEN is_active_order = 1 THEN items_revenue END), 2) AS revenue,
           ROUND(SUM(CASE WHEN is_active_order = 1 THEN items_revenue END) / SUM(is_active_order), 2) AS aov,
           ROUND(100 * AVG(is_late), 2)               AS late_pct,
           ROUND(AVG(CASE WHEN is_late = 1 THEN delay_days END), 2) AS avg_delay_late_only,
           ROUND(AVG(review_score), 3)                AS avg_review
    FROM vw_pbi_orders
"""))

vw_pbi_orders            99441
vw_pbi_order_category    97062
vw_pbi_seller            3095
dim_state                27


,orders,active_orders,revenue,aov,late_pct,avg_delay_late_only,avg_review
0,99441,98199.0,13494400.74,137.42,6.78,10.62,4.086


## 5. Export the four tables to CSV for Power BI

In [5]:
out = ROOT / "powerbi" / "data"
out.mkdir(parents=True, exist_ok=True)
for name in ["vw_pbi_orders", "vw_pbi_order_category", "vw_pbi_seller", "dim_state"]:
    df = run(f"SELECT * FROM {name}")
    df.to_csv(out / f"{name}.csv", index=False)
    print(f"saved {name}.csv", df.shape)

saved vw_pbi_orders.csv (99441, 23)
saved vw_pbi_order_category.csv (97062, 6)
saved vw_pbi_seller.csv (3095, 12)
saved dim_state.csv (27, 4)


## 6. Numbers quoted in the README

Late rate by month after March 2018, and Sao Paulo's late rate and share of late orders.

In [ ]:
print("Late rate by purchase month, Mar-Aug 2018")
display(run("""
    SELECT purchase_month, COUNT(*) AS delivered_orders,
           ROUND(100 * AVG(is_late), 1)  AS late_pct,
           ROUND(AVG(delay_days), 2)     AS avg_delay_days
    FROM vw_order_base
    WHERE is_valid_delivery = 1
      AND purchase_month BETWEEN '2018-03-01' AND '2018-08-01'
    GROUP BY purchase_month ORDER BY purchase_month
"""))

print("Sao Paulo (SP): late rate and share of all late orders")
display(run("""
    SELECT customer_state, COUNT(*) AS delivered_orders, SUM(is_late) AS late_orders,
           ROUND(100 * AVG(is_late), 2) AS late_rate_pct,
           ROUND(100 * SUM(is_late) / (SELECT SUM(is_late) FROM vw_order_base WHERE is_valid_delivery = 1), 2) AS share_of_all_late_orders_pct,
           ROUND(100 * COUNT(*) / (SELECT COUNT(*) FROM vw_order_base WHERE is_valid_delivery = 1), 2) AS share_of_delivered_orders_pct
    FROM vw_order_base
    WHERE is_valid_delivery = 1 AND customer_state = 'SP'
    GROUP BY customer_state
"""))

## Findings

- Feed sizes: `vw_pbi_orders` 99,441 rows, `vw_pbi_order_category` 97,062, `vw_pbi_seller` 3,095, `dim_state` 27.
- Reference values (canceled and unavailable orders excluded from revenue): 98,199 active orders, revenue BRL 13,494,400.74, average order value BRL 137.42, late rate 6.78%, average delay of late orders 10.62 days, average review 4.086.